# 1. Why Transform Data?

### Concept & Definition
Data transformation alters the mathematical distribution or scale of feature variables to satisfy algorithmic assumptions (e.g., normality, homoscedasticity, linearity).

### Real-World / Business Example
In financial modeling, income and transaction amounts are severely right-skewed—a few high-earning individuals stretch the tail. Transforming these features compresses extreme values and stabilizes variance.

### ML Impact
Linear models, Logistic Regression, and Neural Networks assume Gaussian (normal) feature distributions. Unskewing features improves gradient convergence and model accuracy.

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt

# Load dataset
df = pd.read_csv("Cleaned_Validated_Data.csv")

# Ensure numeric columns are clean
df["MonthlyCharges"] = pd.to_numeric(df["MonthlyCharges"], errors="coerce").fillna(df["MonthlyCharges"].median())
df["TenureYears"] = pd.to_numeric(df["TenureYears"], errors="coerce").fillna(df["TenureYears"].median())

print("=== Initial Skewness Assessment ===")
print(f"MonthlyCharges Skewness: {df['MonthlyCharges'].skew():.4f}")
print(f"TenureYears Skewness: {df['TenureYears'].skew():.4f}")

=== Initial Skewness Assessment ===
MonthlyCharges Skewness: 0.2984
TenureYears Skewness: 0.5316


# 2. Log Transformation

### Mathematical Formula
$$y = \log(x + 1)$$

### When to Use
Highly right-skewed continuous data where values are strictly non-negative ($x \ge 0$).

### When NOT to Use
Data containing negative values ($x < 0$).

In [2]:
# Log Transformation
df_trans = df.copy()
df_trans["MonthlyCharges_Log"] = np.log1p(df_trans["MonthlyCharges"])

print("Original MonthlyCharges Skew:", df_trans["MonthlyCharges"].skew().round(4))
print("Log Transformed MonthlyCharges Skew:", df_trans["MonthlyCharges_Log"].skew().round(4))

Original MonthlyCharges Skew: 0.2984
Log Transformed MonthlyCharges Skew: -0.1637


# 3. Square Root Transformation

### Mathematical Formula
$$y = \sqrt{x}$$

### When to Use
Moderately right-skewed count data or non-negative features.

### Advantages & Limitations
- **Advantage:** Weaker than log transformation; preserves zeroes naturally without offset adjustments.
- **Limitation:** Cannot handle negative numbers.

In [3]:
# Square Root Transformation
df_trans["TenureYears_Sqrt"] = np.sqrt(df_trans["TenureYears"])

print("Original TenureYears Skew:", df_trans["TenureYears"].skew().round(4))
print("Sqrt Transformed TenureYears Skew:", df_trans["TenureYears_Sqrt"].skew().round(4))

Original TenureYears Skew: 0.5316
Sqrt Transformed TenureYears Skew: -0.2935


# 4. Box-Cox Transformation

### Mathematical Formula
$$y^{(\lambda)} = \begin{cases} \frac{x^\lambda - 1}{\lambda} & \text{if } \lambda \neq 0 \\ \log(x) & \text{if } \lambda = 0 \end{cases}$$

### When to Use
Strictly positive numerical data ($x > 0$) requiring automatic selection of the optimal variance-stabilizing parameter $\lambda$.

In [4]:
# Box-Cox Transformation (Requires strictly positive values > 0)
charges_pos = df_trans["MonthlyCharges"].apply(lambda x: x if x > 0 else 0.01)
df_trans["MonthlyCharges_BoxCox"], fitted_lambda = stats.boxcox(charges_pos)

print(f"Optimal Box-Cox Lambda (λ): {fitted_lambda:.4f}")
print("Box-Cox Transformed Skew:", pd.Series(df_trans["MonthlyCharges_BoxCox"]).skew().round(4))

Optimal Box-Cox Lambda (λ): 0.2447
Box-Cox Transformed Skew: -0.0492


# 5. Yeo-Johnson Transformation

### Concept & Definition
An extension of the Box-Cox transformation that handles zero and negative numbers seamlessly without requiring prior manual value shifts.

In [5]:
# Yeo-Johnson Transformation
df_trans["MonthlyCharges_YJ"], yj_lambda = stats.yeojohnson(df_trans["MonthlyCharges"])

print(f"Optimal Yeo-Johnson Lambda (λ): {yj_lambda:.4f}")
print("Yeo-Johnson Transformed Skew:", pd.Series(df_trans["MonthlyCharges_YJ"]).skew().round(4))

Optimal Yeo-Johnson Lambda (λ): 0.2362
Yeo-Johnson Transformed Skew: -0.0466


# 6. Distribution Comparison Before & After Transformation

### Evaluation Metrics
- **Skewness:** Measure of asymmetry (Target: close to $0$).
- **Kurtosis:** Measure of tail heaviness (Target: close to $3$ or excess kurtosis $\approx 0$).

In [6]:
# Summary comparison table
comp_df = pd.DataFrame({
    "Original": [df_trans["MonthlyCharges"].skew(), df_trans["MonthlyCharges"].kurt()],
    "Log1p": [df_trans["MonthlyCharges_Log"].skew(), df_trans["MonthlyCharges_Log"].kurt()],
    "Box-Cox": [pd.Series(df_trans["MonthlyCharges_BoxCox"]).skew(), pd.Series(df_trans["MonthlyCharges_BoxCox"]).kurt()],
    "Yeo-Johnson": [pd.Series(df_trans["MonthlyCharges_YJ"]).skew(), pd.Series(df_trans["MonthlyCharges_YJ"]).kurt()]
}, index=["Skewness", "Kurtosis"])

print("=== Distribution Summary Comparison ===")
display(comp_df.round(4))
print("\nNotebook 09 execution completed successfully!")

=== Distribution Summary Comparison ===


,Original,Log1p,Box-Cox,Yeo-Johnson
Skewness,0.2984,-0.1637,-0.0492,-0.0466
Kurtosis,-1.3463,-1.2539,-1.2839,-1.2845



Notebook 09 execution completed successfully!
